In [ ]:
import os
import math
import random
import time
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


# ============================================================
# Repro / device
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def reset_cuda_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def get_cuda_peak_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0


# ============================================================
# Dataset choice
# ============================================================
DATASET_CONFIGS = {
    "mnist": {
        "name": "MNIST",
        "num_classes": 10,
        "img_size": 28,
        "in_chans": 1,
        "mean": (0.1307,),
        "std": (0.3081,),
        "train_split": 55000,
        "val_split": 5000,
        "epochs": 80,
        "crop_padding": 2,
        "hflip": False,
        "patch_size": 7,
        "window_size": 2,
        "num_slices": 4,
    },
    "cifar10": {
        "name": "CIFAR-10",
        "num_classes": 10,
        "img_size": 32,
        "in_chans": 3,
        "mean": (0.4914, 0.4822, 0.4465),
        "std": (0.2470, 0.2435, 0.2616),
        "train_split": 45000,
        "val_split": 5000,
        "epochs": 150,
        "crop_padding": 4,
        "hflip": True,
        "patch_size": 4,
        "window_size": 4,
        "num_slices": 4,
    },
    "cifar100": {
        "name": "CIFAR-100",
        "num_classes": 100,
        "img_size": 32,
        "in_chans": 3,
        "mean": (0.5071, 0.4867, 0.4408),
        "std": (0.2675, 0.2565, 0.2761),
        "train_split": 45000,
        "val_split": 5000,
        "epochs": 150,
        "crop_padding": 4,
        "hflip": True,
        "patch_size": 4,
        "window_size": 4,
        "num_slices": 4,
    },
}


def ask_dataset():
    print("Choose dataset: mnist / cifar10 / cifar100")
    ds = input("Dataset to train/test: ").strip().lower()

    if ds not in DATASET_CONFIGS:
        raise ValueError(f"Unknown dataset '{ds}'. Use one of: {list(DATASET_CONFIGS.keys())}")

    return ds, DATASET_CONFIGS[ds]


DATASET_KEY, CFG = ask_dataset()
NUM_CLASSES = CFG["num_classes"]


# ============================================================
# 2D sinusoidal positional embedding
# ============================================================
def build_2d_sincos_position_embedding(embed_dim: int, grid_size: int, device=None):
    assert embed_dim % 4 == 0
    device = device or torch.device("cpu")

    y = torch.arange(grid_size, device=device, dtype=torch.float32)
    x = torch.arange(grid_size, device=device, dtype=torch.float32)
    yy, xx = torch.meshgrid(y, x, indexing="ij")

    yy = yy.reshape(-1)
    xx = xx.reshape(-1)

    dim_each = embed_dim // 2
    dim_half = dim_each // 2

    omega = torch.arange(dim_half, device=device, dtype=torch.float32)
    omega = 1.0 / (10000 ** (omega / dim_half))

    y_phase = yy[:, None] * omega[None, :]
    x_phase = xx[:, None] * omega[None, :]

    pe_y = torch.cat([torch.sin(y_phase), torch.cos(y_phase)], dim=1)
    pe_x = torch.cat([torch.sin(x_phase), torch.cos(x_phase)], dim=1)

    pe = torch.cat([pe_y, pe_x], dim=1)
    return pe.unsqueeze(0)


# ============================================================
# Metrics
# ============================================================
@torch.no_grad()
def update_confusion_matrix(confmat, preds, targets, num_classes):
    k = targets * num_classes + preds
    confmat.view(-1).index_add_(0, k, torch.ones_like(k, dtype=confmat.dtype))


@torch.no_grad()
def macro_prf1_from_confmat(confmat, eps=1e-12):
    tp = confmat.diag()
    fp = confmat.sum(0) - tp
    fn = confmat.sum(1) - tp

    precision_c = tp / (tp + fp + eps)
    recall_c = tp / (tp + fn + eps)
    f1_c = 2 * precision_c * recall_c / (precision_c + recall_c + eps)

    return precision_c.mean().item(), recall_c.mean().item(), f1_c.mean().item()


# ============================================================
# Window partitioning
# ============================================================
def whr_window_partition(x, window_size: int):
    B, H, W, C = x.shape
    assert H % window_size == 0 and W % window_size == 0

    x = x.view(
        B,
        H // window_size,
        window_size,
        W // window_size,
        window_size,
        C,
    )

    windows = x.permute(0, 1, 3, 2, 4, 5)
    windows = windows.contiguous().view(-1, window_size, window_size, C)
    return windows


def whr_window_reverse(windows, window_size: int, H: int, W: int, B: int):
    C = windows.shape[-1]

    x = windows.view(
        B,
        H // window_size,
        W // window_size,
        window_size,
        window_size,
        C,
    )

    x = x.permute(0, 1, 3, 2, 4, 5)
    x = x.contiguous().view(B, H, W, C)
    return x


# ============================================================
# Fixed reservoir bank
# ============================================================
class FixedReservoirBank(nn.Module):
    def __init__(
        self,
        num_reservoirs,
        input_dim,
        reservoir_dim,
        output_dim,
        noise_std,
        bulk_weight,
        cycle_weight,
        jump_weight,
        spectral_scale,
        jump_size,
        phi_sq_scale=0.1,
        squash_feedback=True,
        noise_in_eval=False,
    ):
        super().__init__()

        self.num_reservoirs = int(num_reservoirs)
        self.input_dim = int(input_dim)
        self.reservoir_dim = int(reservoir_dim)
        self.output_dim = int(output_dim)

        self.noise_std = float(noise_std)
        self.bulk_weight = float(bulk_weight)
        self.cycle_weight = float(cycle_weight)
        self.jump_weight = float(jump_weight)
        self.spectral_scale = float(spectral_scale)

        self.jump_size = max(2, min(int(jump_size), reservoir_dim - 1))
        self.phi_sq_scale = float(phi_sq_scale)
        self.squash_feedback = bool(squash_feedback)
        self.noise_in_eval = bool(noise_in_eval)

        W_res_t, V_in = self._make_reservoir_bank()

        self.W_res_t = nn.Parameter(W_res_t, requires_grad=False)
        self.V_in = nn.Parameter(V_in, requires_grad=False)

    def _make_single_reservoir(self):
        N = self.reservoir_dim

        W = torch.full((N, N), self.bulk_weight, dtype=torch.float32)

        W[0, N - 1] = self.cycle_weight
        for q in range(1, N):
            W[q, q - 1] = self.cycle_weight

        current = 0
        while (current + self.jump_size) < N:
            nxt = current + self.jump_size
            W[current, nxt] = self.jump_weight
            W[nxt, current] = self.jump_weight
            current = nxt

        i = torch.randint(0, N, (1,)).item()
        j = torch.randint(0, N, (1,)).item()

        while j == i:
            j = torch.randint(0, N, (1,)).item()

        W[i, j] = 0.0
        W[j, i] = 0.0

        signs = torch.where(torch.rand(N, N) < 0.5, -1.0, 1.0)
        W = W * signs

        rho = torch.max(torch.abs(torch.linalg.eigvals(W))).real
        W = self.spectral_scale * (W / (rho + 1e-12))

        V = (torch.rand(N, self.input_dim) * 2.0 - 1.0) * 0.1
        mask = (torch.rand(N, self.input_dim) < 0.1).float()
        V = V * mask

        return W.T.contiguous(), V

    def _make_reservoir_bank(self):
        W_list = []
        V_list = []

        for _ in range(self.num_reservoirs):
            Wt, V = self._make_single_reservoir()
            W_list.append(Wt)
            V_list.append(V)

        return torch.stack(W_list, dim=0), torch.stack(V_list, dim=0)

    def _closed_loop_features(self, u_t, x_t, y_prev):
        g = self.phi_sq_scale
        return torch.cat(
            [
                u_t,
                x_t,
                y_prev,
                g * u_t * u_t,
                g * x_t * x_t,
                g * y_prev * y_prev,
            ],
            dim=-1,
        )

    def forward(self, u_seq, warmup_steps, shared_readout, pad_warmup=True):
        B, R, T, Din = u_seq.shape

        assert R == self.num_reservoirs
        assert Din == self.input_dim

        dev = u_seq.device

        x_t = torch.zeros(B, R, self.reservoir_dim, device=dev)
        y_t = torch.zeros(B, R, self.output_dim, device=dev)

        outputs = []

        if pad_warmup:
            dummy = torch.zeros(B, R, warmup_steps, Din, device=dev)
            u_ext = torch.cat([dummy, u_seq], dim=2)
            total_steps = T + warmup_steps
        else:
            u_ext = u_seq
            total_steps = T

        use_noise = self.noise_std > 0 and (self.training or self.noise_in_eval)

        for t in range(total_steps):
            u_t = u_ext[:, :, t, :]

            x_in = torch.einsum("brd,rnd->brn", u_t, self.V_in)
            x_rec = torch.einsum("brn,rmn->brm", x_t, self.W_res_t)

            if use_noise:
                eps_t = torch.randn(B, R, self.reservoir_dim, device=dev) * self.noise_std
            else:
                eps_t = 0.0

            x_t = torch.tanh(x_in + x_rec + eps_t)

            y_feedback = torch.tanh(y_t) if self.squash_feedback else y_t
            phi_t = self._closed_loop_features(u_t, x_t, y_feedback)

            y_t = shared_readout(phi_t.reshape(B * R, -1))
            y_t = y_t.view(B, R, self.output_dim)

            if (not pad_warmup) or (t >= warmup_steps):
                outputs.append(y_t)

        return torch.stack(outputs, dim=2)


# ============================================================
# Hierarchical slice-and-mix reservoir
# ============================================================
class HierarchicalSliceMixReservoir(nn.Module):
    def __init__(
        self,
        embed_dim,
        reservoir_dim,
        num_slices,
        warmup,
        noise_std,
        bulk_weight,
        cycle_weight,
        jump_weight,
        spectral_scale,
        jump_size,
        phi_sq_scale=0.1,
        squash_feedback=True,
        use_slice_residual=True,
        noise_in_eval=False,
    ):
        super().__init__()

        assert num_slices >= 1

        self.embed_dim = int(embed_dim)
        self.reservoir_dim = int(reservoir_dim)
        self.num_slices = int(num_slices)
        self.warmup = int(warmup)
        self.use_slice_residual = bool(use_slice_residual)

        self.num_directions = 4
        self.slice_summary_dim = 3 * self.embed_dim

        self.stage1_slice_reservoirs = FixedReservoirBank(
            num_reservoirs=self.num_directions * self.num_slices,
            input_dim=embed_dim,
            reservoir_dim=reservoir_dim,
            output_dim=embed_dim,
            noise_std=noise_std,
            bulk_weight=bulk_weight,
            cycle_weight=cycle_weight,
            jump_weight=jump_weight,
            spectral_scale=spectral_scale,
            jump_size=jump_size,
            phi_sq_scale=phi_sq_scale,
            squash_feedback=squash_feedback,
            noise_in_eval=noise_in_eval,
        )

        self.stage2_mixing_reservoirs = FixedReservoirBank(
            num_reservoirs=self.num_directions,
            input_dim=self.slice_summary_dim,
            reservoir_dim=reservoir_dim,
            output_dim=embed_dim,
            noise_std=noise_std,
            bulk_weight=bulk_weight,
            cycle_weight=cycle_weight,
            jump_weight=jump_weight,
            spectral_scale=spectral_scale,
            jump_size=jump_size,
            phi_sq_scale=phi_sq_scale,
            squash_feedback=squash_feedback,
            noise_in_eval=noise_in_eval,
        )

        stage1_phi_dim = 2 * (reservoir_dim + 2 * embed_dim)
        self.stage1_closed_loop_readout = nn.Linear(stage1_phi_dim, embed_dim)

        stage2_phi_dim = 2 * (self.slice_summary_dim + reservoir_dim + embed_dim)
        self.stage2_closed_loop_readout = nn.Linear(stage2_phi_dim, embed_dim)

        nn.init.xavier_uniform_(self.stage1_closed_loop_readout.weight, gain=0.01)
        nn.init.zeros_(self.stage1_closed_loop_readout.bias)

        nn.init.xavier_uniform_(self.stage2_closed_loop_readout.weight, gain=0.01)
        nn.init.zeros_(self.stage2_closed_loop_readout.bias)

    def _slice_summary(self, stage1_tokens):
        start_token = stage1_tokens[:, :, :, 0, :]
        end_token = stage1_tokens[:, :, :, -1, :]
        mean_token = stage1_tokens.mean(dim=3)

        return torch.cat([start_token, end_token, mean_token], dim=-1)

    def forward(self, directional_tokens):
        B, num_dirs, T, D = directional_tokens.shape

        assert num_dirs == 4
        assert D == self.embed_dim
        assert T % self.num_slices == 0

        slice_len = T // self.num_slices

        slices = directional_tokens.view(B, 4, self.num_slices, slice_len, D)
        stage1_input = slices.view(B, 4 * self.num_slices, slice_len, D)

        stage1_output = self.stage1_slice_reservoirs(
            stage1_input,
            warmup_steps=self.warmup,
            shared_readout=self.stage1_closed_loop_readout,
            pad_warmup=True,
        )

        stage1_output = stage1_output.view(B, 4, self.num_slices, slice_len, D)

        summaries = self._slice_summary(stage1_output)

        mixed_slices = self.stage2_mixing_reservoirs(
            summaries,
            warmup_steps=self.warmup,
            shared_readout=self.stage2_closed_loop_readout,
            pad_warmup=True,
        )

        mixed_token_level = mixed_slices.unsqueeze(3)
        mixed_token_level = mixed_token_level.expand(B, 4, self.num_slices, slice_len, D)

        if self.use_slice_residual:
            out = stage1_output + mixed_token_level
        else:
            out = mixed_token_level

        return out.contiguous().view(B, 4, T, D)


# ============================================================
# Windowed Hierarchical Reservoir block
# ============================================================
class WindowedHierarchicalReservoirBlock(nn.Module):
    def __init__(
        self,
        embed_dim,
        reservoir_dim,
        warmup,
        noise_std,
        bulk_weight,
        cycle_weight,
        jump_weight,
        spectral_scale,
        jump_size,
        window_size=4,
        shift_size=0,
        num_slices=4,
        phi_sq_scale=0.1,
        squash_feedback=True,
        use_slice_residual=True,
        noise_in_eval=False,
    ):
        super().__init__()

        assert window_size >= 1
        assert 0 <= shift_size < window_size

        self.embed_dim = int(embed_dim)
        self.window_size = int(window_size)
        self.shift_size = int(shift_size)
        self.num_slices = int(num_slices)

        tokens_per_window = self.window_size * self.window_size
        assert tokens_per_window % self.num_slices == 0

        self.slice_mix_reservoir = HierarchicalSliceMixReservoir(
            embed_dim=embed_dim,
            reservoir_dim=reservoir_dim,
            num_slices=num_slices,
            warmup=warmup,
            noise_std=noise_std,
            bulk_weight=bulk_weight,
            cycle_weight=cycle_weight,
            jump_weight=jump_weight,
            spectral_scale=spectral_scale,
            jump_size=jump_size,
            phi_sq_scale=phi_sq_scale,
            squash_feedback=squash_feedback,
            use_slice_residual=use_slice_residual,
            noise_in_eval=noise_in_eval,
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.residual_ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def _four_directional_scan(self, window_tokens, ws):
        Bwin, T, D = window_tokens.shape

        assert T == ws * ws

        grid = window_tokens.view(Bwin, ws, ws, D)

        seq_lr = grid.reshape(Bwin, T, D)
        seq_rl = grid.flip(dims=(2,)).reshape(Bwin, T, D)

        seq_tb = grid.permute(0, 2, 1, 3).contiguous()
        seq_tb = seq_tb.reshape(Bwin, T, D)

        seq_bt = grid.permute(0, 2, 1, 3).flip(dims=(2,)).contiguous()
        seq_bt = seq_bt.reshape(Bwin, T, D)

        directional_tokens = torch.stack([seq_lr, seq_rl, seq_tb, seq_bt], dim=1)
        directional_outputs = self.slice_mix_reservoir(directional_tokens)

        out_lr = directional_outputs[:, 0].view(Bwin, ws, ws, D)

        out_rl = directional_outputs[:, 1].view(Bwin, ws, ws, D)
        out_rl = out_rl.flip(dims=(2,))

        out_tb = directional_outputs[:, 2].view(Bwin, ws, ws, D)
        out_tb = out_tb.permute(0, 2, 1, 3)

        out_bt = directional_outputs[:, 3].view(Bwin, ws, ws, D)
        out_bt = out_bt.flip(dims=(2,)).permute(0, 2, 1, 3)

        out = torch.stack([out_lr, out_rl, out_tb, out_bt], dim=0).mean(dim=0)
        return out.view(Bwin, T, D)

    def forward(self, patch_tokens):
        B, T, D = patch_tokens.shape

        grid_size = int(math.sqrt(T))
        assert grid_size * grid_size == T

        ws = self.window_size
        assert grid_size % ws == 0

        x = patch_tokens.view(B, grid_size, grid_size, D)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        windows = whr_window_partition(x, ws)
        window_tokens = windows.view(-1, ws * ws, D)

        mixed_window_tokens = self._four_directional_scan(window_tokens, ws)

        mixed_windows = mixed_window_tokens.view(-1, ws, ws, D)
        y = whr_window_reverse(mixed_windows, ws, grid_size, grid_size, B)

        if self.shift_size > 0:
            y = torch.roll(y, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        y = y.view(B, T, D)

        return self.residual_ffn(self.norm(y)) + y


# ============================================================
# HiRo classifier
# ============================================================
class HiRoClassifier(nn.Module):
    def __init__(
        self,
        img_size=32,
        in_chans=3,
        patch_size=4,
        embed_dim=128,
        reservoir_dim=200,
        num_classes=100,
        num_whr_blocks=2,
        warmup=2,
        noise_std=0.02,
        bulk_weight=0.08,
        cycle_weight=0.05,
        jump_weight=0.5,
        spectral_scale=0.9,
        jump_size=137,
        phi_sq_scale=0.1,
        squash_feedback=True,
        use_pos_emb=True,
        window_size=4,
        num_slices=4,
        use_slice_residual=True,
        noise_in_eval=False,
    ):
        super().__init__()

        assert img_size % patch_size == 0
        assert embed_dim % 4 == 0

        self.patch_size = int(patch_size)
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.embed_dim = int(embed_dim)
        self.use_pos_emb = bool(use_pos_emb)

        assert self.grid_size % window_size == 0
        assert (window_size * window_size) % num_slices == 0

        self.patch_projection = nn.Linear(patch_size * patch_size * in_chans, embed_dim)
        self.patch_norm = nn.LayerNorm(embed_dim)

        if use_pos_emb:
            pe = build_2d_sincos_position_embedding(
                embed_dim,
                self.grid_size,
                device=torch.device("cpu"),
            )
            self.register_buffer("pos_emb", pe, persistent=False)
        else:
            self.pos_emb = None

        self.whr_blocks = nn.ModuleList(
            [
                WindowedHierarchicalReservoirBlock(
                    embed_dim=embed_dim,
                    reservoir_dim=reservoir_dim,
                    warmup=warmup,
                    noise_std=noise_std,
                    bulk_weight=bulk_weight,
                    cycle_weight=cycle_weight,
                    jump_weight=jump_weight,
                    spectral_scale=spectral_scale,
                    jump_size=jump_size,
                    window_size=window_size,
                    shift_size=0 if (i % 2 == 0) else (window_size // 2),
                    num_slices=num_slices,
                    phi_sq_scale=phi_sq_scale,
                    squash_feedback=squash_feedback,
                    use_slice_residual=use_slice_residual,
                    noise_in_eval=noise_in_eval,
                )
                for i in range(num_whr_blocks)
            ]
        )

        self.final_norm = nn.LayerNorm(embed_dim)

        self.prediction_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        P = self.patch_size

        patches = x.unfold(2, P, P).unfold(3, P, P)
        patches = patches.contiguous().view(B, C, -1, P, P)
        patches = patches.permute(0, 2, 1, 3, 4).flatten(2)

        patch_tokens = self.patch_norm(self.patch_projection(patches))

        if self.use_pos_emb:
            patch_tokens = patch_tokens + self.pos_emb.to(patch_tokens.device)

        for block in self.whr_blocks:
            patch_tokens = block(patch_tokens)

        image_feature = patch_tokens.mean(dim=1)
        image_feature = self.final_norm(image_feature)

        return self.prediction_head(image_feature)


# ============================================================
# Data
# ============================================================
def make_dataloaders(cfg):
    train_tfms = [
        transforms.RandomCrop(cfg["img_size"], padding=cfg["crop_padding"]),
    ]

    if cfg["hflip"]:
        train_tfms.append(transforms.RandomHorizontalFlip())

    train_tfms += [
        transforms.ToTensor(),
        transforms.Normalize(cfg["mean"], cfg["std"]),
    ]

    test_tfms = [
        transforms.ToTensor(),
        transforms.Normalize(cfg["mean"], cfg["std"]),
    ]

    transform_train = transforms.Compose(train_tfms)
    transform_test = transforms.Compose(test_tfms)

    if DATASET_KEY == "mnist":
        train_full = datasets.MNIST(
            root="./data",
            train=True,
            download=True,
            transform=transform_train,
        )

        train_ds, val_ds = random_split(
            train_full,
            [cfg["train_split"], cfg["val_split"]],
            generator=torch.Generator().manual_seed(0),
        )

        val_ds.dataset = datasets.MNIST(
            root="./data",
            train=True,
            download=False,
            transform=transform_test,
        )

        test_ds = datasets.MNIST(
            root="./data",
            train=False,
            download=True,
            transform=transform_test,
        )

    elif DATASET_KEY == "cifar10":
        train_full = datasets.CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform=transform_train,
        )

        train_ds, val_ds = random_split(
            train_full,
            [cfg["train_split"], cfg["val_split"]],
            generator=torch.Generator().manual_seed(0),
        )

        val_ds.dataset = datasets.CIFAR10(
            root="./data",
            train=True,
            download=False,
            transform=transform_test,
        )

        test_ds = datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
            transform=transform_test,
        )

    elif DATASET_KEY == "cifar100":
        train_full = datasets.CIFAR100(
            root="./data",
            train=True,
            download=True,
            transform=transform_train,
        )

        train_ds, val_ds = random_split(
            train_full,
            [cfg["train_split"], cfg["val_split"]],
            generator=torch.Generator().manual_seed(0),
        )

        val_ds.dataset = datasets.CIFAR100(
            root="./data",
            train=True,
            download=False,
            transform=transform_test,
        )

        test_ds = datasets.CIFAR100(
            root="./data",
            train=False,
            download=True,
            transform=transform_test,
        )

    else:
        raise ValueError(DATASET_KEY)

    train_loader = DataLoader(
        train_ds,
        batch_size=128,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=128,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=128,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_dataloaders(CFG)


# ============================================================
# Train / eval
# ============================================================
def run_epoch(model, loader, optimizer, criterion, train_mode, num_classes):
    model.train(train_mode)

    total_loss = 0.0
    total_correct = 0
    total_n = 0

    confmat = torch.zeros(num_classes, num_classes, device=device, dtype=torch.float32)

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(x)
            loss = criterion(logits, y)

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        preds = logits.argmax(dim=1)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_correct += (preds == y).sum().item()
        total_n += bs

        update_confusion_matrix(confmat, preds, y, num_classes)

    precision, recall, f1 = macro_prf1_from_confmat(confmat)

    return (
        total_loss / total_n,
        total_correct / total_n,
        precision,
        recall,
        f1,
    )


# ============================================================
# Efficiency
# ============================================================
@torch.no_grad()
def measure_forward_time(model, loader, warmup_batches=10, max_batches=None):
    model.eval()

    it = iter(loader)

    for _ in range(min(warmup_batches, len(loader))):
        try:
            x, _ = next(it)
        except StopIteration:
            break

        x = x.to(device, non_blocking=True)

        sync_cuda()
        _ = model(x)
        sync_cuda()

    total_time = 0.0
    total_samples = 0
    total_batches = 0

    for bidx, (x, _) in enumerate(loader):
        if max_batches is not None and bidx >= max_batches:
            break

        x = x.to(device, non_blocking=True)

        sync_cuda()
        t0 = time.perf_counter()
        _ = model(x)
        sync_cuda()
        t1 = time.perf_counter()

        total_time += t1 - t0
        total_samples += x.size(0)
        total_batches += 1

    sec_per_batch = total_time / max(total_batches, 1)
    ms_per_image = (total_time / max(total_samples, 1)) * 1000.0

    return total_time, sec_per_batch, ms_per_image


@torch.no_grad()
def estimate_trainable_forward_flops(model, input_shape):
    flops = {}

    def linear_hook(name):
        def hook(module, inp, out):
            if not any(p.requires_grad for p in module.parameters()):
                return

            x = inp[0]
            batch_elems = x.numel() // x.shape[-1]

            f = batch_elems * (2 * module.in_features * module.out_features)

            if module.bias is not None:
                f += batch_elems * module.out_features

            flops[name] = flops.get(name, 0) + f

        return hook

    def layernorm_hook(name):
        def hook(module, inp, out):
            if not any(p.requires_grad for p in module.parameters()):
                return

            x = inp[0]
            f = 7 * x.numel()

            flops[name] = flops.get(name, 0) + f

        return hook

    hooks = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(linear_hook(name)))
        elif isinstance(module, nn.LayerNorm):
            hooks.append(module.register_forward_hook(layernorm_hook(name)))

    dummy = torch.randn(*input_shape, device=device)

    was_training = model.training
    model.eval()
    _ = model(dummy)

    for h in hooks:
        h.remove()

    if was_training:
        model.train()

    return sum(flops.values()), flops


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return total, trainable


# ============================================================
# Model
# ============================================================
model = HiRoClassifier(
    img_size=CFG["img_size"],
    in_chans=CFG["in_chans"],
    patch_size=CFG["patch_size"],
    embed_dim=128,
    reservoir_dim=200,
    num_classes=CFG["num_classes"],
    num_whr_blocks=2,
    warmup=2,
    noise_std=0.02,
    bulk_weight=0.08,
    cycle_weight=0.05,
    jump_weight=0.5,
    spectral_scale=0.9,
    jump_size=137,
    phi_sq_scale=0.1,
    squash_feedback=True,
    use_pos_emb=True,
    window_size=CFG["window_size"],
    num_slices=CFG["num_slices"],
    use_slice_residual=True,
    noise_in_eval=False,
).to(device)


total_params, trainable_params = count_parameters(model)

trainable_flops_per_image, flops_breakdown = estimate_trainable_forward_flops(
    model,
    input_shape=(1, CFG["in_chans"], CFG["img_size"], CFG["img_size"]),
)


# ============================================================
# Optimizer / schedule
# ============================================================
BASE_LR = 3e-4
MIN_LR = 1e-5
EPOCHS = CFG["epochs"]
WARMUP_EPOCHS = 2

optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=BASE_LR,
    weight_decay=1e-5,
)

criterion = nn.CrossEntropyLoss()


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)

    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

    return (MIN_LR / BASE_LR) + (1.0 - (MIN_LR / BASE_LR)) * cosine


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


# ============================================================
# Run
# ============================================================
best_val_acc = 0.0
best_epoch = 0
best_state = None

print("\n============================================================")
print(f"HiRo / WHR run on {CFG['name']}")
print("============================================================")
print(f"device: {device}")
print(f"params: trainable={trainable_params:,} | total={total_params:,}")
print(f"approx trainable forward FLOPs/image: {trainable_flops_per_image:,}")
print("------------------------------------------------------------")

for epoch in range(1, EPOCHS + 1):
    sync_cuda()
    reset_cuda_peak()
    epoch_t0 = time.perf_counter()

    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        train_mode=True,
        num_classes=NUM_CLASSES,
    )

    sync_cuda()
    train_peak_mb = get_cuda_peak_mb()

    sync_cuda()
    val_t0 = time.perf_counter()

    va_loss, va_acc, va_p, va_r, va_f1 = run_epoch(
        model,
        val_loader,
        optimizer,
        criterion,
        train_mode=False,
        num_classes=NUM_CLASSES,
    )

    sync_cuda()
    val_loop_time = time.perf_counter() - val_t0

    scheduler.step()
    cur_lr = optimizer.param_groups[0]["lr"]

    sync_cuda()
    epoch_time = time.perf_counter() - epoch_t0

    val_forward_time, val_sec_batch, val_ms_img = measure_forward_time(
        model,
        val_loader,
        warmup_batches=5,
        max_batches=None,
    )

    print(
        f"ep {epoch:03d} | "
        f"lr {cur_lr:.2e} | "
        f"time {epoch_time:.2f}s | "
        f"peak {train_peak_mb:.1f} MB | "
        f"train {tr_loss:.4f}/{tr_acc:.4f}/{tr_p:.4f}/{tr_r:.4f}/{tr_f1:.4f} | "
        f"val {va_loss:.4f}/{va_acc:.4f}/{va_p:.4f}/{va_r:.4f}/{va_f1:.4f} | "
        f"val_fwd {val_forward_time:.2f}s/{val_ms_img:.4f} ms-img"
    )

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }


print("------------------------------------------------------------")
print(f"best checkpoint: ep {best_epoch:03d} | val_acc={best_val_acc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)


# ============================================================
# Test
# ============================================================
sync_cuda()
reset_cuda_peak()
test_t0 = time.perf_counter()

te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(
    model,
    test_loader,
    optimizer,
    criterion,
    train_mode=False,
    num_classes=NUM_CLASSES,
)

sync_cuda()
test_loop_time = time.perf_counter() - test_t0
test_peak_mb = get_cuda_peak_mb()

test_forward_time, test_sec_batch, test_ms_img = measure_forward_time(
    model,
    test_loader,
    warmup_batches=10,
    max_batches=None,
)

print("\n============================================================")
print("Final result")
print("============================================================")
print(f"dataset: {CFG['name']}")
print(f"best_epoch: {best_epoch}")
print(f"params_trainable: {trainable_params:,}")
print(f"params_total: {total_params:,}")
print(f"trainable_forward_flops_per_image: {trainable_flops_per_image:,}")
print(f"test_loss: {te_loss:.4f}")
print(f"test_acc: {te_acc:.4f}")
print(f"test_precision_macro: {te_p:.4f}")
print(f"test_recall_macro: {te_r:.4f}")
print(f"test_f1_macro: {te_f1:.4f}")
print(f"test_eval_loop_time_sec: {test_loop_time:.2f}")
print(f"test_forward_only_time_sec: {test_forward_time:.2f}")
print(f"test_forward_sec_per_batch: {test_sec_batch:.4f}")
print(f"test_forward_ms_per_image: {test_ms_img:.4f}")
print(f"test_peak_cuda_memory_mb: {test_peak_mb:.1f}")
print("============================================================")